# Reproduksi Eksperimen SATD (Tabel 4) - Li et al. (2023)
**Paper:** *Automatic identification of self-admitted technical debt from four different sources*  
**Journal:** *Empirical Software Engineering (EMSE 2023)*  
**DOI:** [10.1007/s10664-023-10297-9](https://doi.org/10.1007/s10664-023-10297-9)

---
### Ringkasan Tujuan:
Notebook ini mereproduksi **Tabel 4** dari paper secara mandiri (*from scratch*) hanya dengan modal 4 dataset CSV yang disediakan pada repositori GitHub ([yikun-li/satd-different-sources-data](https://github.com/yikun-li/satd-different-sources-data)).

Tabel 4 mengevaluasi 6 classifier pada 4 sumber data (`Comment`, `Commit`, `Pull`, `Issue`) untuk 4 tipe technical debt (`C/D.`, `DOC.`, `TST.`, `REQ.`):
1. **Deep Learning:** Single-Task Text-CNN & Multitask Text-CNN (MT-Text-CNN) dengan *randomized word vectors* (Kim 2014).
2. **Traditional ML:** Logistic Regression (LR), Support Vector Machine (SVM - default RBF kernel Sklearn), Random Forest (RF) dengan TF-IDF.
3. **Baseline:** Random Classifier berbasis distribusi frekuensi kelas apriori.
4. **Evaluasi:** Stratified 10-Fold Cross-Validation.

In [ ]:
# Cell 1: Import Dependencies & Setup Environment
import os
import re
import sys
import time
import math
import string
import random
import warnings
import numpy as np
import pandas as pd

# Machine Learning & Metrics
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support

# Deep Learning (PyTorch)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# NLP
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

warnings.filterwarnings('ignore')

# Konfigurasi Eksekusi:
# Set QUICK_TEST = True jika hanya ingin menguji 1 fold untuk verifikasi pipeline cepat.
# Set QUICK_TEST = False untuk menjalankan full 10-fold CV persis paper.
QUICK_TEST = False
N_SPLITS = 2 if QUICK_TEST else 10
EPOCHS = 10  # Standar Text-CNN (Kim 2014 & Zhang & Wallace 2017)
BATCH_SIZE = 64

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()
gpu_names = [torch.cuda.get_device_name(i) for i in range(num_gpus)] if num_gpus > 0 else []
print(f'Device: {device} | Total GPU: {num_gpus} {gpu_names} | N_SPLITS: {N_SPLITS} | QUICK_TEST: {QUICK_TEST}')

## 1. Data Preparation & Label Standardization
Sesuai **Bab 3.5 & Tabel 3 paper**:
* Sumber data: `Comment` (Code Comments), `Commit` (Commit Messages), `Pull` (Pull Requests), dan `Issue` (Issue Tracker).
* Standardisasi 5 Kelas Target:
  * `0`: `non-SATD` (Negative class: mencakup non_debt, architecture_debt, build_debt, defect_debt)
  * `1`: `code|design-debt` (C/D: menggabungkan code_debt dan design_debt)
  * `2`: `documentation-debt` (DOC)
  * `3`: `test-debt` (TST)
  * `4`: `requirement-debt` (REQ)

In [ ]:
# Cell 2: Download Dataset dari GitHub (Siap untuk Google Colab) & Standardisasi Label
import urllib.request

GITHUB_REPO_URL = 'https://github.com/yikun-li/satd-different-sources-data.git'
RAW_BASE_URL = 'https://raw.githubusercontent.com/yikun-li/satd-different-sources-data/main'
DATA_DIR = './data'

required_files = [
    'satd-dataset-code_comments.csv',
    'satd-dataset-commit_messages.csv',
    'satd-dataset-pull_requests.csv',
    'satd-dataset-issues.csv'
]

os.makedirs(DATA_DIR, exist_ok=True)
all_present = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in required_files)

if not all_present:
    print('Dataset tidak ditemukan di lokal. Mengunduh langsung dari GitHub (Google Colab Mode)...')
    # Opsi 1: Cepat via git clone
    exit_code = os.system(f'git clone {GITHUB_REPO_URL} /tmp/satd_repo && cp /tmp/satd_repo/*.csv {DATA_DIR}/ && rm -rf /tmp/satd_repo')
    # Opsi 2: Fallback via HTTP download jika git clone gagal
    if exit_code != 0 or not all(os.path.exists(os.path.join(DATA_DIR, f)) for f in required_files):
        for f in required_files:
            dest = os.path.join(DATA_DIR, f)
            if not os.path.exists(dest):
                url = f'{RAW_BASE_URL}/{f}'
                print(f'Mengunduh {f}...')
                urllib.request.urlretrieve(url, dest)
    print('Unduhan dari GitHub selesai!')
else:
    print('Dataset sudah tersedia di folder:', DATA_DIR)

DATA_FILES = {
    'Comment': os.path.join(DATA_DIR, 'satd-dataset-code_comments.csv'),
    'Commit': os.path.join(DATA_DIR, 'satd-dataset-commit_messages.csv'),
    'Pull': os.path.join(DATA_DIR, 'satd-dataset-pull_requests.csv'),
    'Issue': os.path.join(DATA_DIR, 'satd-dataset-issues.csv')
}

def map_label(raw_label):
    raw = str(raw_label).lower().strip()
    if 'code' in raw or 'design' in raw:
        return 1  # C/D.
    elif 'documentation' in raw:
        return 2  # DOC.
    elif 'test' in raw:
        return 3  # TST.
    elif 'requirement' in raw:
        return 4  # REQ.
    else:
        return 0  # non-SATD (Other)

dfs = {}
stats_list = []

for src_name, file_path in DATA_FILES.items():
    if not os.path.exists(file_path):
        raise FileNotFoundError(f'File {file_path} tidak ditemukan!')
    
    df = pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip')
    text_col = 'text' if 'text' in df.columns else df.columns[1]
    cls_col = 'classification' if 'classification' in df.columns else ('label' if 'label' in df.columns else df.columns[-1])
    
    df['clean_label'] = df[cls_col].apply(map_label)
    df['raw_text'] = df[text_col].astype(str)
    
    dfs[src_name] = df[['raw_text', 'clean_label']].copy()
    
    counts = df['clean_label'].value_counts().to_dict()
    stats_list.append({
        'Source': src_name,
        'C/D.': counts.get(1, 0),
        'DOC.': counts.get(2, 0),
        'TST.': counts.get(3, 0),
        'REQ.': counts.get(4, 0),
        'Other (non-SATD)': counts.get(0, 0),
        'Total': len(df)
    })

stats_df = pd.DataFrame(stats_list)
print('=== DISTRIBUSI KELAS DATASET (Verifikasi dengan Tabel 3 Paper) ===')
print(stats_df.to_string(index=False))

## 2. Text Preprocessing
Menggunakan aturan pra-pemrosesan teks dari Bab 3.4 dan script `satd_detector.py`:
1. Menghilangkan karakter pembuka/penutup komentar (`//`, `/*`, `*/`).
2. Menggunakan `TweetTokenizer` untuk memisahkan token.
3. Menghilangkan simbol khusus, angka, dan mengubah semua kata menjadi huruf kecil (*lowercase*).

In [ ]:
# Cell 3: Pra-pemrosesan Teks
tokenizer_words = nltk.TweetTokenizer()
punctuation_chars = string.punctuation.replace('!', '').replace('?', '')

def preprocess_text(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    comment = re.sub(r'(//)|(/\*)|(\*/)', '', text).lower()
    try:
        tokens_sentences = [tokenizer_words.tokenize(t) for t in nltk.sent_tokenize(comment)]
    except Exception:
        tokens_sentences = [tokenizer_words.tokenize(comment)]

    processed = []
    for sentence in tokens_sentences:
        toks = []
        for token in sentence:
            if token == 'non-nls':
                toks.append(token)
                continue
            elif token == ',':
                toks.append('.')
            elif any(c in token for c in [' ', '.', '#', '_', '/', '>', '<', '@']):
                continue
            elif any(c.isdigit() for c in token):
                continue
            else:
                toks.append(token)
        if toks and toks[-1] != '.':
            toks.append('.')
        processed.extend(toks)

    stripped = [w for w in processed if w and (w not in punctuation_chars and ':' not in w and '=' not in w and ')' not in w and '(' not in w)]
    return ' '.join(stripped)

print('Memulai proses pembersihan teks...')
for src_name in dfs:
    dfs[src_name]['processed_text'] = dfs[src_name]['raw_text'].apply(preprocess_text)
    print(f'  {src_name} selesai ({len(dfs[src_name])} data)')

print('\nContoh Pembersihan Teks:')
print('Sebelum:', dfs['Comment']['raw_text'].iloc[0])
print('Sesudah:', dfs['Comment']['processed_text'].iloc[0])

## 3. Evaluasi Protokol: Stratified K-Fold Cross-Validation
Sesuai **Bab 3.6.4 & 3.6.7**:
* Menggunakan **Stratified 10-Fold Cross-Validation** pada masing-masing sumber data untuk menjamin proporsi kelas seimbang di setiap fold.
* Metrik dihitung per kelas target:
  $$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$
* **AVG (Macro F1)**: Rata-rata F1-score dari 4 tipe target debt: $$\text{AVG} = \frac{F1_{\text{C/D}} + F1_{\text{DOC}} + F1_{\text{TST}} + F1_{\text{REQ}}}{4}$$

In [ ]:
# Cell 4: Setup Stratified CV & Helper Metrik
def calculate_f1_metrics(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=[1, 2, 3, 4], zero_division=0)
    return {
        'C/D.': f1[0],
        'DOC.': f1[1],
        'TST.': f1[2],
        'REQ.': f1[3],
        'AVG.': np.mean(f1)
    }

final_results = {}

# Buat partisi CV untuk tiap dataset
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
kfold_splits = {}
for src_name in dfs:
    X = dfs[src_name]['processed_text'].values
    y = dfs[src_name]['clean_label'].values
    kfold_splits[src_name] = list(skf.split(X, y))
print(f'Partisi {N_SPLITS}-Fold siap untuk 4 sumber data.')

## 4. Model 1: Random Classifier (Baseline)
Sesuai **Bab 3.6.2**: Model memprediksi kelas secara acak proporsional terhadap distribusi kemunculan kelas pada data latih (*prior class probability*).

In [ ]:
# Cell 5: Evaluasi Baseline Random Classifier
print('=== MENJALANKAN BASELINE: RANDOM CLASSIFIER ===')
random_res = {}

for src_name in ['Comment', 'Commit', 'Pull', 'Issue']:
    fold_metrics = []
    y_all = dfs[src_name]['clean_label'].values
    
    for train_idx, test_idx in kfold_splits[src_name]:
        y_train = y_all[train_idx]
        y_test = y_all[test_idx]
        
        classes, counts = np.unique(y_train, return_counts=True)
        probs = counts / len(y_train)
        
        y_pred = np.random.choice(classes, size=len(y_test), p=probs)
        fold_metrics.append(calculate_f1_metrics(y_test, y_pred))
        
    avg_m = {metric: np.mean([fm[metric] for fm in fold_metrics]) for metric in ['C/D.', 'DOC.', 'TST.', 'REQ.', 'AVG.']}
    random_res[src_name] = avg_m
    print(f'  {src_name}: AVG F1 = {avg_m["AVG."]:.4f}')

final_results['Random'] = random_res

## 5. Model 2, 3, 4: Traditional Machine Learning (LR, SVM, RF)
Sesuai **Bab 3.6.1**:
* Fitur diekstraksi menggunakan **TF-IDF** (`TfidfVectorizer`).
* Model menggunakan implementasi default `scikit-learn`:
  * **LR**: `LogisticRegression(max_iter=1000)`
  * **SVM**: `SVC()` (implementasi default SVM di Scikit-Learn menggunakan kernel RBF tanpa class weight, menjelaskan F1 rendah pada data imbalanced)
  * **RF**: `RandomForestClassifier(n_estimators=100)`

In [ ]:
# Cell 6: Evaluasi Traditional Machine Learning (LR, SVM, RF)
traditional_models = {
    'LR': lambda: LogisticRegression(max_iter=1000, random_state=42),
    'SVM': lambda: SVC(random_state=42),  # Default Scikit-Learn SVM (RBF kernel)
    'RF': lambda: RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

for clf_name, model_fn in traditional_models.items():
    print(f'=== MENJALANKAN MODEL: {clf_name} ===')
    clf_res = {}
    for src_name in ['Comment', 'Commit', 'Pull', 'Issue']:
        fold_metrics = []
        X_all = dfs[src_name]['processed_text'].values
        y_all = dfs[src_name]['clean_label'].values
        
        for train_idx, test_idx in kfold_splits[src_name]:
            X_train, X_test = X_all[train_idx], X_all[test_idx]
            y_train, y_test = y_all[train_idx], y_all[test_idx]
            
            tfidf = TfidfVectorizer(max_features=10000)
            X_tr_vec = tfidf.fit_transform(X_train)
            X_te_vec = tfidf.transform(X_test)
            
            clf = model_fn()
            clf.fit(X_tr_vec, y_train)
            y_pred = clf.predict(X_te_vec)
            fold_metrics.append(calculate_f1_metrics(y_test, y_pred))
            
        avg_m = {metric: np.mean([fm[metric] for fm in fold_metrics]) for metric in ['C/D.', 'DOC.', 'TST.', 'REQ.', 'AVG.']}
        clf_res[src_name] = avg_m
        print(f'  {src_name} selesai. AVG F1: {avg_m["AVG."]:.4f}')
        
    final_results[clf_name] = clf_res

## 6. Deep Learning Dataset & Vocabulary Setup
Membangun vocabulary dan representasi integer token untuk masukan model Deep Learning PyTorch.
* Menggunakan padding token `<pad>` (index 0) dan out-of-vocabulary `<unk>` (index 1).
* Word Embedding diinisialisasi secara **acak** (*scratch / randomized word vectors*) sesuai Bab 4.1.1.

In [ ]:
# Cell 7: Vocabulary & PyTorch Dataset Builder
class Vocabulary:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word2idx = {'<pad>': 0, '<unk>': 1}
        self.counts = {}

    def build_vocab(self, texts):
        for text in texts:
            for tok in text.split():
                self.counts[tok] = self.counts.get(tok, 0) + 1
        idx = 2
        for tok, cnt in self.counts.items():
            if cnt >= self.min_freq:
                self.word2idx[tok] = idx
                idx += 1

    def __len__(self):
        return len(self.word2idx)

    def numericalize(self, text, max_len=100):
        tokens = text.split()
        indices = [self.word2idx.get(t, 1) for t in tokens[:max_len]]
        if len(indices) < max_len:
            indices += [0] * (max_len - len(indices))
        return indices

# Bangun vocab dari seluruh teks
all_corpus = []
for src in dfs:
    all_corpus.extend(dfs[src]['processed_text'].tolist())

vocab = Vocabulary(min_freq=2)
vocab.build_vocab(all_corpus)
print(f'Total Vocabulary Tokens: {len(vocab)}')

class SATDDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=100):
        self.features = [vocab.numericalize(t, max_len) for t in texts]
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return torch.tensor(self.features[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

## 7. Model 5: Single-Task Text-CNN (Kim 2014)
Sesuai **Bab 3.6.1 & 4.1.1**:
* Arsitektur: `nn.Embedding(vocab_size, 300)` (randomly initialized), `Conv2d` dengan ukuran kernel `[3, 4, 5]`, 100 filters, dan `Dropout(0.5)`.
* Dilatih secara terpisah untuk masing-masing sumber data.

In [ ]:
# Cell 8: Single-Task Text-CNN
class SingleTextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=300, num_classes=5, kernel_sizes=[3, 4, 5], kernel_num=100, dropout=0.5):
        super(SingleTextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv2d(1, kernel_num, (k, embed_dim)) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(len(kernel_sizes) * kernel_num, num_classes)

    def forward(self, x):
        x_embed = self.embedding(x).unsqueeze(1)
        x_conv = [F.relu(conv(x_embed)).squeeze(3) for conv in self.convs]
        x_pool = [F.max_pool1d(c, c.size(2)).squeeze(2) for c in x_conv]
        x_cat = torch.cat(x_pool, 1)
        x_drop = self.dropout(x_cat)
        return self.fc(x_drop)

def train_eval_single_cnn(src_name, epochs=EPOCHS, batch_size=BATCH_SIZE):
    print(f'Training Single Text-CNN: {src_name}')
    fold_metrics = []
    X_all = dfs[src_name]['processed_text'].values
    y_all = dfs[src_name]['clean_label'].values

    for fold, (train_idx, test_idx) in enumerate(kfold_splits[src_name]):
        tr_ds = SATDDataset(X_all[train_idx], y_all[train_idx], vocab)
        te_ds = SATDDataset(X_all[test_idx], y_all[test_idx], vocab)
        tr_loader = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
        te_loader = DataLoader(te_ds, batch_size=batch_size, shuffle=False)

        model = SingleTextCNN(len(vocab)).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        model.train()
        for epoch in range(epochs):
            for bx, by in tr_loader:
                bx, by = bx.to(device), by.to(device)
                optimizer.zero_grad()
                out = model(bx)
                loss = criterion(out, by)
                loss.backward()
                optimizer.step()

        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for bx, by in te_loader:
                bx = bx.to(device)
                pred = torch.argmax(model(bx), dim=1).cpu().numpy()
                preds.extend(pred)
                trues.extend(by.numpy())

        m = calculate_f1_metrics(trues, preds)
        fold_metrics.append(m)
        print(f'  Fold {fold+1}/{N_SPLITS} - AVG F1: {m["AVG."]:.3f}')

    return {metric: np.mean([fm[metric] for fm in fold_metrics]) for metric in ['C/D.', 'DOC.', 'TST.', 'REQ.', 'AVG.']}

text_cnn_res = {}
for src in ['Comment', 'Commit', 'Pull', 'Issue']:
    text_cnn_res[src] = train_eval_single_cnn(src, epochs=EPOCHS, batch_size=BATCH_SIZE)

final_results['Text-CNN'] = text_cnn_res

## 8. Model 6: Multitask Text-CNN (MT-Text-CNN)
Sesuai **Bab 3.6.1 & Bab 3.6.5 (Collobert & Weston 2008)**:
* Shared Encoder: Layer Embedding acak dan Convolutional feature extractor digunakan bersama oleh ke-4 task.
* Task Heads: 4 output linear layer terpisah untuk Comment, Commit, Pull Request, dan Issue.
* Multitask Stochastic Training: Setiap step mengambil mini-batch dari task acak $t \in [0, 1, 2, 3]$ dan melakukan pembaruan gradien.

In [ ]:
# Cell 9: MT-Text-CNN Multitask Architecture & Stochastic Training
class MTTextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=300, num_classes=5, num_tasks=4, kernel_sizes=[3, 4, 5], kernel_num=100, dropout=0.5):
        super(MTTextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv2d(1, kernel_num, (k, embed_dim)) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fcs = nn.ModuleList([
            nn.Linear(len(kernel_sizes) * kernel_num, num_classes) for _ in range(num_tasks)
        ])

    def forward(self, x, task_id):
        x_embed = self.embedding(x).unsqueeze(1)
        x_conv = [F.relu(conv(x_embed)).squeeze(3) for conv in self.convs]
        x_pool = [F.max_pool1d(c, c.size(2)).squeeze(2) for c in x_conv]
        x_cat = torch.cat(x_pool, 1)
        x_drop = self.dropout(x_cat)
        return self.fcs[task_id](x_drop)

def train_eval_mt_cnn(epochs=EPOCHS, batch_size=BATCH_SIZE):
    print(f'=== TRAINING MULTITASK TEXT-CNN (MT-Text-CNN) {N_SPLITS}-FOLD CV ===')
    source_list = ['Comment', 'Commit', 'Pull', 'Issue']
    task_map = {src: i for i, src in enumerate(source_list)}
    fold_results = {src: [] for src in source_list}

    for fold in range(N_SPLITS):
        print(f'\n--- Fold {fold + 1}/{N_SPLITS} MT-Text-CNN ---')
        train_loaders = {}
        test_loaders = {}

        for src in source_list:
            train_idx, test_idx = kfold_splits[src][fold]
            X_all = dfs[src]['processed_text'].values
            y_all = dfs[src]['clean_label'].values
            
            tr_ds = SATDDataset(X_all[train_idx], y_all[train_idx], vocab)
            te_ds = SATDDataset(X_all[test_idx], y_all[test_idx], vocab)
            train_loaders[src] = DataLoader(tr_ds, batch_size=batch_size, shuffle=True)
            test_loaders[src] = DataLoader(te_ds, batch_size=batch_size, shuffle=False)

        model = MTTextCNN(len(vocab)).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.CrossEntropyLoss()

        # Stochastic Multitask Training
        model.train()
        for epoch in range(epochs):
            iterators = {src: iter(train_loaders[src]) for src in source_list}
            finished = {src: False for src in source_list}

            while not all(finished.values()):
                avail = [s for s in source_list if not finished[s]]
                if not avail:
                    break
                task = random.choice(avail)
                task_id = task_map[task]

                try:
                    bx, by = next(iterators[task])
                except StopIteration:
                    finished[task] = True
                    continue

                bx, by = bx.to(device), by.to(device)
                optimizer.zero_grad()
                out = model(bx, task_id)
                loss = criterion(out, by)
                loss.backward()
                optimizer.step()

        # Evaluasi task per source
        model.eval()
        with torch.no_grad():
            for src in source_list:
                preds, trues = [], []
                task_id = task_map[src]
                for bx, by in test_loaders[src]:
                    bx = bx.to(device)
                    pred = torch.argmax(model(bx, task_id), dim=1).cpu().numpy()
                    preds.extend(pred)
                    trues.extend(by.numpy())
                
                m = calculate_f1_metrics(trues, preds)
                fold_results[src].append(m)
                print(f'  {src} Fold {fold+1} AVG F1: {m["AVG."]:.3f}')

    mt_res = {}
    for src in source_list:
        mt_res[src] = {metric: np.mean([fm[metric] for fm in fold_results[src]]) for metric in ['C/D.', 'DOC.', 'TST.', 'REQ.', 'AVG.']}
    return mt_res

final_results['MT-Text-CNN'] = train_eval_mt_cnn(epochs=EPOCHS, batch_size=BATCH_SIZE)

## 9. Final Output: Format Tabel 4 (Sesuai Paper)
Mencetak seluruh hasil eksperimen dalam tabel rekapitulasi identik dengan **Tabel 4** di jurnal *Empirical Software Engineering 2023*.

In [ ]:
# Cell 10: Format dan Cetak Output Tabel 4 Sesuai Paper
def print_table_4(results):
    rows = []
    
    sections = [
        ('Deep Learning', [
            ('Text-CNN', results.get('Text-CNN', {})),
            ('MT-Text-CNN', results.get('MT-Text-CNN', {}))
        ]),
        ('Traditional Machine Learning', [
            ('LR', results.get('LR', {})),
            ('SVM', results.get('SVM', {})),
            ('RF', results.get('RF', {}))
        ]),
        ('Baseline', [
            ('Random', results.get('Random', {}))
        ])
    ]
    
    rand_dict = results.get('Random', {})
    
    for sec_name, models in sections:
        rows.append([f'-- {sec_name} --', '', '', '', '', '', '', ''])
        for model_name, model_data in models:
            first = True
            for debt in ['C/D.', 'DOC.', 'TST.', 'REQ.', 'AVG.']:
                c = model_data.get('Comment', {}).get(debt, 0.0)
                m = model_data.get('Commit', {}).get(debt, 0.0)
                p = model_data.get('Pull', {}).get(debt, 0.0)
                i = model_data.get('Issue', {}).get(debt, 0.0)
                avg_val = np.mean([c, m, p, i])
                
                # Random baseline average for improvement ratio
                rand_avg = np.mean([
                    rand_dict.get(s, {}).get(debt, 0.001) for s in ['Comment', 'Commit', 'Pull', 'Issue']
                ])
                imp = f'{avg_val / rand_avg:.1f}x' if (rand_avg > 0 and model_name != 'Random') else '-'
                
                row_label = model_name if first else ''
                first = False
                
                rows.append([
                    row_label,
                    debt,
                    f'{c:.3f}',
                    f'{m:.3f}',
                    f'{p:.3f}',
                    f'{i:.3f}',
                    f'{avg_val:.3f}',
                    imp
                ])
    
    headers = ['Classifier', 'Type of SATD', 'Comment', 'Commit', 'Pull', 'Issue', 'Avg.', 'Average Imp. over Random']
    print(pd.DataFrame(rows, columns=headers).to_string(index=False))

print('\n' + '='*85)
print('Table 4: Comparison of F1-score between machine learning and baseline approaches')
print('='*85)
print_table_4(final_results)